#### Code pour faire marcher PySpark chez moi 

In [1]:
import os, subprocess, shutil
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
print("winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot" 
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ";" + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("java in PATH:", shutil.which("java"))
print(subprocess.run(["java","-version"], capture_output=True, text=True).stderr)

winutils: True
hadoop.dll: True
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
java in PATH: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot\bin\java.EXE
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)



In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

import pyspark.pandas as ps
ps.set_option("compute.fail_on_ansi_mode", False)

print("ansi =", spark.conf.get("spark.sql.ansi.enabled"))

ansi = false


# Code principal tests

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyspark.pandas as ps
from pyspark.sql.functions import *

In [4]:
mission_paris = pd.read_csv(".\data\BDD_BGES\BDD_BGES_PARIS\BDD_BGES_PARIS_MISSION\MISSION_20260429.txt", sep=";", index_col="ID_MISSION")

In [11]:
mission_paris["VILLE_DEPART"].unique()

array(['Paris', 'Shanghai'], dtype=object)

In [4]:
pers_paris = pd.read_csv(".\data\BDD_BGES\BDD_BGES_PARIS\PERSONNEL_PARIS.txt", sep=";", index_col="ID_PERSONNEL")

In [5]:
pers_paris.head()

,NOM_PERSONNEL,PRENOM_PERSONNEL,DT_NAISS,VILLE_NAISS,PAYS_NAISS,NUM_SECU,IND_PAYS_NUM_TELP,NUM_TELEPHONE,NUM_VOIE,DSC_VOIE,CMPL_VOIE,CD_POSTAL,VILLE,PAYS,FONCTION_PERSONNEL,TS_CREATION_PERSONNEL,TS_MAJ_PPERSONNEL
ID_PERSONNEL,,,,,,,,,,,,,,,,,
KeyPers_Paris_1230000,Name0,FistName0,1931-07-27,Tokyo,Japan,NS000000000,NaN,+336##0014965,28,NomVoie510,NaN,#7168,Paris,France,Cadre,2009-03-23 14:28:33,2009-03-23 14:28:33
KeyPers_Paris_1230001,Name1,FistName1,1963-01-07,Melbourne,Australia,NS000000001,NaN,+336##0743025,60,NomVoie171,NaN,#9979,Paris,France,DRH,2017-05-10 07:55:58,2017-05-10 07:55:58
KeyPers_Paris_1230002,Name2,FistName2,1979-04-03,Sidney,Australia,NS000000002,NaN,+336##0098564,73,NomVoie382,NaN,#9505,Paris,France,Ingénieur Data,2002-11-23 16:12:12,2002-11-23 16:12:12
KeyPers_Paris_1230003,Name3,FistName3,1944-08-01,Montreal,Canada,NS000000003,NaN,+336##0171838,36,NomVoie323,NaN,#0725,Paris,France,Ingénieur Informaticien,1999-01-15 12:34:15,1999-01-15 12:34:15
KeyPers_Paris_1230004,Name4,FistName4,1981-01-21,Lille,France,NS000000004,NaN,+336##0570970,69,NomVoie554,NaN,#8557,Paris,France,Ingénieur Informaticien,2020-01-04 22:34:38,2020-01-04 22:34:38


#### Readfile : charge les fichiers en spark sql dataframe

In [14]:
def readFile(path, indexCol):
    return ps.read_csv(path, sep=';', index_col=indexCol)

In [15]:
mat_info_paris_psdf = readFile(".\data\BDD_BGES\BDD_BGES_PARIS\BDD_BGES_PARIS_INFORMATIQUE\MATERIEL_INFORMATIQUE_20261103.txt", "ID_MATERIELINFO")
mat_info_paris_psdf.head()

,ID_PERSONNEL,NOM_PERSONNEL,PRENOM_PERSONNEL,DATE_ACHAT,TYPE,MODELE
ID_MATERIELINFO,,,,,,
Paris_MATERIEL_INFO_202611030,KeyPers_Paris_1233148,Name3148,FistName3148,2026-11-03 16:38:16,Tablette,modèle par défaut
Paris_MATERIEL_INFO_202611031,KeyPers_Paris_1230105,Name105,FistName105,2026-11-03 12:07:08,PC fixe tout-en-un,
Paris_MATERIEL_INFO_202611032,KeyPers_Paris_1231526,Name1526,FistName1526,2026-11-03 09:42:50,PC fixe sans ecran,"Prodesk (Tower, SFF)"
Paris_MATERIEL_INFO_202611033,KeyPers_Paris_1233104,Name3104,FistName3104,2026-11-03 11:15:58,Disque dur,modèle par défaut
Paris_MATERIEL_INFO_202611034,KeyPers_Paris_1233731,Name3731,FistName3731,2026-11-03 16:34:22,PC portable,EliteBook 8xx


In [17]:
mat_info_paris_sdf = mat_info_paris_psdf.to_spark()
mat_info_paris_sdf.show()

+--------------------+-------------+----------------+-------------------+------------------+--------------------+
|        ID_PERSONNEL|NOM_PERSONNEL|PRENOM_PERSONNEL|         DATE_ACHAT|              TYPE|              MODELE|
+--------------------+-------------+----------------+-------------------+------------------+--------------------+
|KeyPers_Paris_123...|     Name3148|    FistName3148|2026-11-03 16:38:16|          Tablette|   modèle par défaut|
|KeyPers_Paris_123...|      Name105|     FistName105|2026-11-03 12:07:08|PC fixe tout-en-un|                    |
|KeyPers_Paris_123...|     Name1526|    FistName1526|2026-11-03 09:42:50|PC fixe sans ecran|Prodesk (Tower, SFF)|
|KeyPers_Paris_123...|     Name3104|    FistName3104|2026-11-03 11:15:58|        Disque dur|   modèle par défaut|
|KeyPers_Paris_123...|     Name3731|    FistName3731|2026-11-03 16:34:22|       PC portable|       EliteBook 8xx|
|KeyPers_Paris_123...|     Name4566|    FistName4566|2026-11-03 13:35:32|PC fixe tout-en

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


#### Homogénéisation de la langue des fonctions

In [20]:
pers_london_psdf = readFile("./data/BDD_BGES/BDD_BGES_LONDON/PERSONNEL_LONDON.txt", "ID_PERSONNEL")
pers_london_psdf["FONCTION_PERSONNEL"].unique()

0     Computer Engineer
1                   HRD
2    Business Executive
3             Economist
4         Data Engineer
Name: FONCTION_PERSONNEL, dtype: object

In [22]:
pers_berlin_psdf = readFile("./data/BDD_BGES/BDD_BGES_BERLIN/PERSONNEL_BERLIN.txt", "ID_PERSONNEL")
pers_berlin_psdf["FONCTION_PERSONNEL"].unique()

0               Ökonom
1        Führungskraft
2       Personalleiter
3    Computeringenieur
4       Dateningenieur
Name: FONCTION_PERSONNEL, dtype: object

In [23]:
pers_paris_psdf = readFile("data/BDD_BGES/BDD_BGES_PARIS/PERSONNEL_PARIS.txt", "ID_PERSONNEL")
pers_paris_psdf["FONCTION_PERSONNEL"].unique()

0    Ingénieur Informaticien
1             Ingénieur Data
2                 Economiste
3                        DRH
4                      Cadre
Name: FONCTION_PERSONNEL, dtype: object

In [24]:
# Comme l'anglais est la langue de 4 sites, on homogeneise en anglais pour avoir moins d'opérations à faire
def clean_langue_fonction(psdf, site):
    match site:
        case "BERLIN":
            map_fonctions = {
                "Ökonom": "Economist",
                "Führungskraft": "Business Executive",
                "Personalleiter": "HRD",
                "Computeringenieur": "Computer Engineer",
                "Dateningenieur": "Data Engineer",
            }
            psdf["FONCTION_PERSONNEL"] = psdf["FONCTION_PERSONNEL"].replace(map_fonctions)
        case "PARIS":
            map_fonctions = {
                "Ingénieur Informaticien": "Computer Engineer",
                "Ingénieur Data": "Data Engineer",
                "Economiste": "Economist",
                "DRH": "HRD",
                "Cadre": "Business Executive",
            }
            psdf["FONCTION_PERSONNEL"] = psdf["FONCTION_PERSONNEL"].replace(map_fonctions)


Test

In [25]:
clean_langue_fonction(pers_paris_psdf, "PARIS")
pers_paris_psdf["FONCTION_PERSONNEL"].unique()

0     Computer Engineer
1                   HRD
2    Business Executive
3             Economist
4         Data Engineer
Name: FONCTION_PERSONNEL, dtype: object

In [26]:
clean_langue_fonction(pers_berlin_psdf, "BERLIN")
pers_paris_psdf["FONCTION_PERSONNEL"].unique()

0     Computer Engineer
1                   HRD
2    Business Executive
3             Economist
4         Data Engineer
Name: FONCTION_PERSONNEL, dtype: object

La langue est bien mise en anglais partout

#### Homogénéisation de la langue des types de missions

In [27]:
mission_londres_psdf = readFile("data/BDD_BGES/BDD_BGES_LONDON/BDD_BGES_LONDON_MISSION/MISSION_20260429.txt", "ID_MISSION")
mission_londres_psdf["TYPE_MISSION"].unique()

0       Business Meeting
1             Conference
2           Team Meeting
3            Development
4    Vocational Training
Name: TYPE_MISSION, dtype: object

In [29]:
mission_berlin_psdf = readFile("data/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BERLIN_MISSION/MISSION_20260429.txt", "ID_MISSION")
mission_berlin_psdf["TYPE_MISSION"].unique()

0    Geschäftstreffen
1           Konferenz
2            Schulung
3             Meeting
4         Entwicklung
Name: TYPE_MISSION, dtype: object

In [30]:
mission_paris_psdf = readFile("data/BDD_BGES/BDD_BGES_PARIS/BDD_BGES_PARIS_MISSION/MISSION_20260429.txt", "ID_MISSION")
mission_paris_psdf["TYPE_MISSION"].unique()

0               Conférence
1                Formation
2                  Réunion
3    Rencontre entreprises
4            Développement
Name: TYPE_MISSION, dtype: object

In [31]:
def clean_langue_mission(psdf, site):
    match site:
        case "BERLIN":
            map_type_mission = {
                "Geschäftstreffen": "Business Meeting",
                "Konferenz": "Conference",
                "Schulung": "Vocational Training",
                "Meeting": "Team Meeting",
                "Entwicklung": "Development",
            }
            psdf["TYPE_MISSION"] = psdf["TYPE_MISSION"].replace(map_type_mission)
        case "PARIS":
            map_type_mission = {
                "Conférence": "Conference",
                "Formation": "Vocational Training",
                "Réunion": "Team Meeting",
                "Rencontre entreprises": "Business Meeting",
                "Développement": "Development",
            }
            psdf["TYPE_MISSION"] = psdf["TYPE_MISSION"].replace(map_type_mission)

Test

In [33]:
clean_langue_mission(mission_paris_psdf, "PARIS")
mission_paris_psdf["TYPE_MISSION"].unique()

0       Business Meeting
1             Conference
2           Team Meeting
3            Development
4    Vocational Training
Name: TYPE_MISSION, dtype: object

#### Test de réponse à une question (à revoir)

Combien d’ingénieurs informaticiens travaillent sur le site de Paris ?

In [ ]:
pers_paris_psdf = readFiles("data/BDD_BGES/BDD_BGES_PARIS", "ID_PERSONNEL")
pers_paris_sdf = pers_paris_psdf.to_spark(index_col="ID_PERSONNEL")

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [ ]:
nb_inge_info = (
    pers_paris_sdf
    .filter(col("FONCTION_PERSONNEL") == "Ingénieur Informaticien")
    .count()
)

print(f"{nb_inge_info} ingénieurs informaticiens travaillent sur le site de Paris")

1873 ingénieurs informaticiens travaillent sur le site de Paris
